## Лабораторная работа № 1
## Разведочный анализ больших данных с Apache Spark

### Часть 1: загрузка, очистка и сохранение в Apache Iceberg

Набор данных: **NYC Yellow Taxi Trip Data** (Kaggle). Подставьте свой путь в HDFS в переменную path.

Подключаем библиотеки.

In [ ]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf


def create_spark_configuration() -> SparkConf:
    """Cozdaet i konfiguriruet ekzemplyar SparkConf dlya prilozheniya Spark."""
    user_name = os.getenv("USER")
    conf = SparkConf()
    conf.setAppName("SOBD lab1 - NYC Taxi EDA")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    return conf

from pyspark.sql.functions import col, lit, to_timestamp, unix_timestamp

Создаём конфигурацию и сессию Spark.

In [ ]:
conf = create_spark_configuration()

In [ ]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Указываем путь к файлам в HDFS (замените USERNAME на своё имя пользователя).

In [ ]:
path = "hdfs:///user/USERNAME/datasets/nyc_taxi/*.csv"

In [ ]:
df = (spark.read.format("csv")
      .option("header", "true")
      .load(path)
)

In [ ]:
df.show()

In [ ]:
df.printSchema()

Оставляем информативные столбцы.

In [ ]:
df = df.select(
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
    "trip_distance", "RatecodeID", "payment_type", "fare_amount",
    "tip_amount", "tolls_amount", "total_amount", "PULocationID", "DOLocationID"
)

Приводим столбцы к корректным типам и добавляем длительность поездки (мин).

In [ ]:
def transform_dataframe(data: DataFrame) -> DataFrame:
    """Privodit stolbcy k nuzhnym tipam i dobavlyaet trip_duration_min."""
    int_cols = ["passenger_count", "RatecodeID", "payment_type", "PULocationID", "DOLocationID"]
    dbl_cols = ["trip_distance", "fare_amount", "tip_amount", "tolls_amount", "total_amount"]
    for c in int_cols:
        data = data.withColumn(c, col(c).cast("Integer"))
    for c in dbl_cols:
        data = data.withColumn(c, col(c).cast("Double"))
    data = data.withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime")))
    data = data.withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))
    data = data.withColumn(
        "trip_duration_min",
        (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60.0,
    )
    return data

In [ ]:
df = transform_dataframe(df)
df.printSchema()

In [ ]:
df.show()

Сохраняем в таблицу Apache Iceberg. Базу называем по фамилии студента.

In [ ]:
database_name = "shahulov_database"

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}")

In [ ]:
spark.catalog.setCurrentDatabase(database_name)

In [ ]:
df.writeTo("sobd_lab1_table").using("iceberg").create()

In [ ]:
for table in spark.catalog.listTables():
    print(table.name)

При необходимости пересоздания таблицы:

In [ ]:
# spark.sql("DROP TABLE spark_catalog.shahulov_database.sobd_lab1_table")
# spark.sql("DROP DATABASE spark_catalog.shahulov_database")

In [ ]:
spark.stop()